# Colab Experiment: General Feature-space Cloaking

Goal: apply ResNet-50 feature-space cloaking, measure feature shift, then run a classification proxy train-after-poison benchmark.

## 1. Setup Colab / GitHub repo

This cell clones the repo when running from a fresh Colab runtime and installs dependencies.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ngocvuq4/adversarial-data-protection.git"
PROJECT_DIR = "adversarial-data-protection"

# If this notebook is opened directly in Colab, clone the GitHub repo first.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("src").exists():
    if not Path(PROJECT_DIR).exists():
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

print("Working directory:", os.getcwd())
REQ_FILE = "requirements-colab.txt" if IN_COLAB else "requirements.txt"
print("Installing:", REQ_FILE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", REQ_FILE], check=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Optional Google Drive output

GitHub stores source code only. Use Drive if you want results to persist after the Colab runtime resets.

In [ ]:
# Google Drive dataset/results paths.
# Your Drive folder is: MyDrive/adversarial-data-protection/
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/adversarial-data-protection"
DRIVE_DATA_ROOT = f"{DRIVE_PROJECT_DIR}/data"
DRIVE_RESULTS_DIR = f"{DRIVE_PROJECT_DIR}/results"

DATA_ROOT = "./data"
RESULTS_ROOT = "./results"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_ROOT = DRIVE_DATA_ROOT
        RESULTS_ROOT = DRIVE_RESULTS_DIR
        Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
        Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
    except ImportError:
        print("Not running in Colab; using local ./data and ./results")

print("DATA_ROOT:", DATA_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


## 3. Run experiment

In [ ]:
import os
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from scripts.run_experiment import setup_dirs, collect_clean_tensors, train_classifier, build_protected_tensors
from src.datasets import get_cifar10
from src.evaluation import compute_attack_success_rate, compute_linf, compute_psnr, compute_ssim
from src.models import evaluate, get_surrogate_resnet50, get_victim_resnet18
from src.visualization import plot_before_after

# Smoke defaults. Cloaking is heavier because CIFAR is upscaled to 224x224 for ResNet-50 PGD.
SUBSET_SIZE = 500
BATCH_SIZE = 64
EPSILON = 0.05
BASELINE_EPOCHS = 2
VICTIM_EPOCHS = 2
PGD_STEPS = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
setup_dirs()

train_loader, test_loader = get_cifar10(root=DATA_ROOT, subset_size=SUBSET_SIZE, batch_size=BATCH_SIZE)
clean_x, clean_y = collect_clean_tensors(train_loader)

print("Training clean baseline victim...")
baseline = train_classifier(lambda: get_victim_resnet18(device=device), train_loader, BASELINE_EPOCHS, device)
baseline_acc = evaluate(baseline, test_loader, device)
print("baseline_clean_test_accuracy:", baseline_acc)

print("Loading ResNet-50 surrogate and generating general feature-space cloaked data...")
surrogate = get_surrogate_resnet50(device)
protected_x = build_protected_tensors(
    "general_cloaking",
    clean_x,
    clean_y,
    device,
    epsilon=EPSILON,
    surrogate=surrogate,
    pgd_steps=PGD_STEPS,
)

@torch.no_grad()
def compute_feature_shift(x_orig, x_protected, batch_size=64):
    cos_values, l2_values = [], []
    surrogate.eval()
    for start in range(0, x_orig.size(0), batch_size):
        xo = F.interpolate(x_orig[start:start+batch_size].to(device), size=(224, 224), mode="bilinear", align_corners=False)
        xp = F.interpolate(x_protected[start:start+batch_size].to(device), size=(224, 224), mode="bilinear", align_corners=False)
        fo = F.normalize(surrogate(xo).float(), dim=1)
        fp = F.normalize(surrogate(xp).float(), dim=1)
        cos_values.append((fo * fp).sum(dim=1).cpu())
        l2_values.append(torch.norm(fp - fo, dim=1).cpu())
    cos = torch.cat(cos_values)
    l2 = torch.cat(l2_values)
    return round(cos.mean().item(), 4), round(l2.mean().item(), 4)

feature_cosine, feature_l2_shift = compute_feature_shift(clean_x, protected_x)

protected_loader = DataLoader(TensorDataset(protected_x, clean_y), batch_size=BATCH_SIZE, shuffle=True)
print("Training victim on cloaked train set...")
victim = train_classifier(lambda: get_victim_resnet18(device=device), protected_loader, VICTIM_EPOCHS, device)
clean_acc, asr = compute_attack_success_rate(victim, test_loader, device)

metrics = {
    "technique": "general_cloaking",
    "victim_model": "ResNet-18",
    "subset_size": SUBSET_SIZE,
    "epsilon": EPSILON,
    "baseline_clean_test_accuracy": baseline_acc,
    "protected_clean_test_accuracy": clean_acc,
    "accuracy_drop": round(baseline_acc - clean_acc, 4),
    "asr_proxy": asr,
    "feature_cosine_original_protected": feature_cosine,
    "feature_l2_shift": feature_l2_shift,
    "psnr": compute_psnr(clean_x, protected_x),
    "ssim": compute_ssim(clean_x, protected_x),
    "linf": compute_linf(clean_x, protected_x),
}
print(metrics)

os.makedirs("results/tables", exist_ok=True)
pd.DataFrame([metrics]).to_csv("results/tables/general_cloaking_experiment.csv", index=False)

sample_idx = 0
plot_before_after(clean_x[sample_idx], protected_x[sample_idx], "general_cloaking")
to_pil = torchvision.transforms.ToPILImage()
to_pil(clean_x[sample_idx]).save("results/protected_samples/general_cloaking_original.png")
to_pil(protected_x[sample_idx]).save("results/protected_samples/general_cloaking_protected.png")
print("Saved results/tables/general_cloaking_experiment.csv and sample images.")

## 4. Optional copy results to Drive

In [ ]:
# Copy local results to Drive results folder if needed.
# Most experiment code writes to ./results first; this keeps a persistent copy in Drive.
if USE_GOOGLE_DRIVE:
    import shutil
    target = Path(RESULTS_ROOT)
    target.mkdir(parents=True, exist_ok=True)
    shutil.copytree("results", target, dirs_exist_ok=True)
    print("Copied local ./results to", target)
